# Caption and alt-text a batch of product images and a short video with Azure Content Understanding

Your media team's catalog has 200 product photos with no alt text and a marketing video pending an accessibility review in two weeks. The accessibility lawyer flagged the catalog last quarter and gave the team a soft deadline. You have one session to stand up a Content Understanding pipeline that captions a batch of 5 sample images in one call, falls back to GPT-4o for any low-confidence image, runs the same pipeline over a 10-second product clip, and produces structured JSON output ready to hand to the front-end team for the alt-text-attribute rollout.

What you will build:

- A Foundry hub plus project in `eastus`, plus an Azure AI Services multi-service resource exposing Content Understanding.
- A Content Understanding analyzer schema with `caption: string`, `alt_text: string`, and `confidence: float` output fields.
- A batch call that captions 5 lab-shipped product images in one request.
- A confidence-based fallback to GPT-4o multimodal for any image scoring below 0.7. (For the lab to exercise the fallback path consistently, the threshold is bumped to 0.99 in code; production should use 0.7.)
- The same pipeline over a 10-second video, producing per-shot caption plus alt-text plus confidence segments.
- A combined structured-JSON artifact with `source` and `media_type` fields distinguishing every record.

**Time.** Plan for about 55 minutes hands-on. Foundry provisioning is 5 to 8 minutes; the AI Services resource is fast; image batch is 15-30 seconds; video is up to 60 seconds. Budget up to 90 minutes total.

**Cost.** Multimodal labs cost a bit more than text-only because image tokenization pulls GPT-4o into the loop. Content Understanding itself is cheap, about a tenth of a cent per image. The fallback path on GPT-4o is the line item to watch: each image input is roughly a thousand tokens whether the photo is a sunset or a screenshot, so debugging the fallback in a tight loop adds up faster than the text labs do. Still under thirty cents end to end. Coffee remains the dominant daily expense.

This lab maps to AI-103 Domain 3: Implement computer vision solutions (13% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Azure SDKs.
# - azure-ai-projects 2.0 (March 2026) absorbed azure-ai-agents.
# - openai>=2.0.0 for the GPT-4o multimodal image-input path.
# - requests for the analyzer HTTP API (Content Understanding is exposed
#   as a REST surface on the AI Services account's data plane).

!pip install --quiet labrun-checks[azure]==0.7.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0 openai>=2.0.0 requests>=2.31.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import getpass
import json
import random
import string
import time

import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.core.exceptions import (
    HttpResponseError,
    ResourceNotFoundError,
    ClientAuthenticationError,
)
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Account, Sku
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from openai import AzureOpenAI

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)
from labrun_checks.adapters.azure import AzureCleanupAdapter

LAB_ID = "content-understanding-multimodal"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
CHAT_DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4o"
ANALYZER_ID = "labrun-content-understanding-analyzer-v1"

# Content Understanding API version. Pin explicitly; preview-era versions
# vary in payload shape.
CONTENT_UNDERSTANDING_API_VERSION = "2024-12-01-preview"

# AI Services account name is globally unique; append a random suffix.
_rand_suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=6))
AI_SERVICES_NAME = f"labrun-content-understanding-multi-{_rand_suffix}"

# GPT-4o for the multimodal fallback path. The capability gap between
# GPT-4o-mini and GPT-4o matters for image inputs; the lab uses 4o.
CHAT_MODEL_NAME = "gpt-4o"
CHAT_MODEL_VERSION = "2024-08-06"
CHAT_MODEL_CAPACITY_TPM = 10  # 10k TPM Standard is the new-account default for 4o

# Force-fallback threshold: real product photos often score above 0.7, so
# the lab uses 0.99 to guarantee at least one fallback dispatch.
CONFIDENCE_THRESHOLD = 0.99

# Resolved during setup.
SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
AOAI_ACCOUNT_ENDPOINT = None
AI_SERVICES_ENDPOINT = None
AZURE_CREDENTIAL = None

# Lab-shipped sample asset URLs. Hosted in the public labrun-labs repo so
# Content Understanding can fetch them. If these change the lab breaks; the
# setup cell HEAD-checks each URL.
SAMPLE_IMAGE_URLS = [
    "https://raw.githubusercontent.com/labrun-labs/assets/main/azure-ai-103/lab-08/img1.jpg",
    "https://raw.githubusercontent.com/labrun-labs/assets/main/azure-ai-103/lab-08/img2.jpg",
    "https://raw.githubusercontent.com/labrun-labs/assets/main/azure-ai-103/lab-08/img3.jpg",
    "https://raw.githubusercontent.com/labrun-labs/assets/main/azure-ai-103/lab-08/img4.jpg",
    "https://raw.githubusercontent.com/labrun-labs/assets/main/azure-ai-103/lab-08/img5.jpg",
]
SAMPLE_VIDEO_URL = "https://raw.githubusercontent.com/labrun-labs/assets/main/azure-ai-103/lab-08/clip.mp4"

# State populated by tasks.
IMAGE_RECORDS = []
VIDEO_RECORDS = []
COMBINED_ARTIFACT = []

# Pricing for wallet checks.
CHAT_PRICE_PER_1M_INPUT_USD = 2.50
CHAT_PRICE_PER_1M_OUTPUT_USD = 10.00
CONTENT_UNDERSTANDING_PER_IMAGE_USD = 0.001
CONTENT_UNDERSTANDING_PER_VIDEO_SECOND_USD = 0.001

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials per
# LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported. Content Understanding GA is in")
    print(f"eastus, westus2, swedencentral, southeastasia. This lab pins {REGION}.")
    raise SystemExit(1)
if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    raise SystemExit(1)
except StopIteration:
    pass

# HEAD-check the sample image and video URLs so a broken asset surfaces here,
# not mid-lab inside a Content Understanding call.
print("HEAD-checking the sample asset URLs...")
for url in SAMPLE_IMAGE_URLS + [SAMPLE_VIDEO_URL]:
    try:
        h = requests.head(url, allow_redirects=True, timeout=10)
        if h.status_code >= 400:
            print(f"  WARNING: {url} returned {h.status_code}. The lab may still work if")
            print(f"           Content Understanding fetches via a different path.")
    except requests.RequestException as e:
        print(f"  WARNING: HEAD {url} failed: {e}. Continuing anyway.")

AZURE_CREDENTIALS_BAG = {"subscription_id": SUBSCRIPTION_ID, "region": REGION, "resource_group": RESOURCE_GROUP}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")
print(f"AI Services account name: {AI_SERVICES_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and Resource Graph orphan scan.
# No critical resources: AI Services and GPT-4o Standard deployments do not
# bill at zero traffic; Content Understanding is per-call only.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="aoai_deployment",
        id=CHAT_DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} --name <attached-aoai-account> "
            f"--deployment-name {CHAT_DEPLOYMENT_NAME}"
        ),
        extra={"account_name": AOAI_ACCOUNT_NAME},
    ),
    CleanupResource(
        type="ai_services_account",
        id=AI_SERVICES_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account delete "
            f"--resource-group {RESOURCE_GROUP} --name {AI_SERVICES_NAME}"
        ),
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print(f"Run the cleanup cell first, or: az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Provision the AI Services multi-service resource and register the Content Understanding analyzer

The Content Understanding API lives on the data plane of an Azure AI Services account (`kind=AIServices`), not a stand-alone resource. You create the account, then PUT a named analyzer schema describing the output fields you want every image and video to produce.

Build it in this order:

1. Create the resource group, hub, project, and `gpt-4o` deployment (the multimodal fallback path uses GPT-4o, not GPT-4o-mini, because image-input quality matters here).
2. Create the AI Services account via `CognitiveServicesManagementClient.accounts.begin_create_or_update` with `kind=AIServices` and `sku.name=S0`.
3. Read the analyzer-schema dict in the cell. It declares `caption: string`, `alt_text: string`, and `confidence: float`.
4. PUT the analyzer to the data-plane endpoint: `PUT {endpoint}/contentunderstanding/analyzers/{ANALYZER_ID}?api-version={version}`. Authenticate with the AAD token from `DefaultAzureCredential`.

**Why `kind=AIServices` and not `kind=OpenAI` or `kind=ComputerVision`.** AIServices is the unified multimodal endpoint that exposes Content Understanding alongside Speech, Translator, and Vision. Standalone Computer Vision is the older surface; standalone OpenAI does not expose Content Understanding at all.

In [ ]:
# NBVAL_SKIP
# Task 1: Provision Foundry, the AI Services resource, and register the
# Content Understanding analyzer.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP, {"location": REGION, "tags": lab_tags},
)
print(f"Resource group ready: {RESOURCE_GROUP}")

hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": HUB_NAME, "public_network_access": "Enabled"},
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
hub_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
).result()

project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {"friendly_name": PROJECT_NAME, "hub_resource_id": hub_workspace.id},
}
print("Watching the project workspace go through provisioning, 2 to 3 minutes more...")
project_workspace = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
).result()

for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
aoai_account = cs_client.accounts.get(RESOURCE_GROUP, AOAI_ACCOUNT_NAME)
AOAI_ACCOUNT_ENDPOINT = aoai_account.properties.endpoint
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")

deployment_payload = {
    "sku": {"name": "Standard", "capacity": CHAT_MODEL_CAPACITY_TPM},
    "properties": {
        "model": {"format": "OpenAI", "name": CHAT_MODEL_NAME, "version": CHAT_MODEL_VERSION},
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}
print("Watching the GPT-4o deployment go through Succeeded purgatory...")
cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, CHAT_DEPLOYMENT_NAME, deployment_payload,
).result()
print(f"GPT-4o deployment ready: {CHAT_DEPLOYMENT_NAME}")

# AI Services account for Content Understanding.
ai_services_payload = Account(
    location=REGION,
    tags=lab_tags,
    kind="AIServices",
    sku=Sku(name="S0"),
    properties={"custom_sub_domain_name": AI_SERVICES_NAME},
    identity={"type": "SystemAssigned"},
)
print(f"Provisioning AI Services multi-service resource: {AI_SERVICES_NAME}...")
# YOUR CODE: Provision the AI Services account via
# ai_services_account = cs_client.accounts.begin_create_or_update(
#     RESOURCE_GROUP, AI_SERVICES_NAME, ai_services_payload,
# ).result()
# Store the result in ai_services_account.

AI_SERVICES_ENDPOINT = ai_services_account.properties.endpoint
print(f"AI Services account ready. Endpoint: {AI_SERVICES_ENDPOINT}")

# Register the analyzer schema via the data-plane REST API. The Python SDK
# does not yet ship a Content Understanding client; the HTTP path is the
# canonical interaction surface.
analyzer_schema = {
    "description": "Caption and alt-text analyzer for product images and short videos",
    "scenario": "image",
    "outputSchema": {
        "type": "object",
        "properties": {
            "caption": {
                "type": "string",
                "description": "A concise descriptive caption of the visual content.",
            },
            "alt_text": {
                "type": "string",
                "description": "Accessibility alt-text optimised for screen readers.",
            },
            "confidence": {
                "type": "number",
                "description": "Model confidence in the caption and alt-text, 0 to 1.",
            },
        },
        "required": ["caption", "alt_text", "confidence"],
    },
}

aad_token = AZURE_CREDENTIAL.get_token("https://cognitiveservices.azure.com/.default").token
analyzer_url = (
    f"{AI_SERVICES_ENDPOINT}contentunderstanding/analyzers/{ANALYZER_ID}"
    f"?api-version={CONTENT_UNDERSTANDING_API_VERSION}"
)
print(f"Registering analyzer schema at {analyzer_url}")
analyzer_response = requests.put(
    analyzer_url,
    headers={"Authorization": f"Bearer {aad_token}", "Content-Type": "application/json"},
    json=analyzer_schema,
    timeout=30,
)
if analyzer_response.status_code >= 400:
    print(f"Analyzer registration failed: {analyzer_response.status_code}")
    print(analyzer_response.text)
    raise SystemExit(1)
print(f"Analyzer registered: {ANALYZER_ID}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: AI Services account exists with kind=AIServices and the
# lab tag; the Content Understanding analyzer exists and exposes the
# expected output schema (caption: string, alt_text: string).

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        cs = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
        try:
            acct = cs.accounts.get(RESOURCE_GROUP, AI_SERVICES_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AI Services account {AI_SERVICES_NAME} does not exist in "
                    f"{RESOURCE_GROUP}. Re-run Task 1."
                ),
            )

        if (acct.kind or "").lower() != "aiservices":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Account kind is {acct.kind!r}, expected 'AIServices'. Standalone "
                    f"ComputerVision or OpenAI accounts do not expose Content Understanding."
                ),
            )
        if (acct.location or "").lower() != REGION:
            return CheckpointResult(
                status="fail",
                error_reason=f"Account location is {acct.location!r}, expected {REGION!r}.",
            )
        tags = acct.tags or {}
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Account is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. Found: {tags}"
                ),
            )
        prov = getattr(acct.properties, "provisioning_state", None) if acct.properties else None
        if prov and str(prov).lower() != "succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=f"provisioning_state is {prov!r}, expected 'Succeeded'.",
            )

        # Fetch the analyzer via the data-plane REST API.
        token = AZURE_CREDENTIAL.get_token("https://cognitiveservices.azure.com/.default").token
        url = (
            f"{AI_SERVICES_ENDPOINT}contentunderstanding/analyzers/{ANALYZER_ID}"
            f"?api-version={CONTENT_UNDERSTANDING_API_VERSION}"
        )
        resp = requests.get(
            url, headers={"Authorization": f"Bearer {token}"}, timeout=30,
        )
        if resp.status_code == 404:
            return CheckpointResult(
                status="fail",
                error_reason=f"Analyzer {ANALYZER_ID} not found. Did the PUT call succeed?",
            )
        if resp.status_code >= 400:
            return CheckpointResult(
                status="fail",
                error_reason=f"Analyzer GET failed: {resp.status_code} {resp.text[:200]}",
            )
        body = resp.json()
        output = body.get("outputSchema") or {}
        props = output.get("properties") or {}
        caption_prop = props.get("caption") or {}
        if caption_prop.get("type") != "string":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"outputSchema.caption.type is {caption_prop.get('type')!r}, expected 'string'."
                ),
            )
        alt_prop = props.get("alt_text") or {}
        if alt_prop.get("type") != "string":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"outputSchema.alt_text.type is {alt_prop.get('type')!r}, expected 'string'."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names the field that is off. If the kind complaint fires, you passed `kind="OpenAI"` or `kind="ComputerVision"` on the payload; switch to `"AIServices"`. If analyzer GET returns 404, the PUT failed with a 4xx that you skipped past.

</details>

<details><summary>Hint 2 (direction)</summary>

One LRO. `ai_services_account = cs_client.accounts.begin_create_or_update(RESOURCE_GROUP, AI_SERVICES_NAME, ai_services_payload).result()`. The `ai_services_payload` already has `kind="AIServices"`, `sku=Sku(name="S0")`, and the `custom_sub_domain_name` set (required so the data-plane endpoint can use AAD auth).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
ai_services_account = cs_client.accounts.begin_create_or_update(
    RESOURCE_GROUP, AI_SERVICES_NAME, ai_services_payload,
).result()
```

</details>

**Wallet check.** Still at $0.00. AI Services creation and the analyzer PUT are control-plane operations, free. Coffee remains the daily expense champion.

## Task 2: Analyze a batch of 5 product images in one Content Understanding call

The analyzer's batch endpoint accepts an array of image URLs and returns one record per image with the declared output schema fields. The lab ships 5 sample image URLs (HEAD-checked in setup); your job is to construct the batch payload and parse the response.

Build it in this order:

1. Compose a POST to `{endpoint}/contentunderstanding/analyzers/{ANALYZER_ID}:analyze?api-version={version}` with a JSON body of `{"inputs": [{"url": ...}, ...]}`.
2. Wait for completion. Content Understanding is async for video but sync for image batches up to 5 URLs; the response returns the per-image records directly.
3. Parse each record's `caption`, `alt_text`, and `confidence`. Build `IMAGE_RECORDS` as a list of dicts with `{"image_url", "caption", "alt_text", "confidence", "source": "content_understanding"}`.

In [ ]:
# NBVAL_SKIP
# Task 2: Batch image analysis via the analyzer's :analyze endpoint.

token = AZURE_CREDENTIAL.get_token("https://cognitiveservices.azure.com/.default").token
analyze_url = (
    f"{AI_SERVICES_ENDPOINT}contentunderstanding/analyzers/{ANALYZER_ID}:analyze"
    f"?api-version={CONTENT_UNDERSTANDING_API_VERSION}"
)
analyze_payload = {
    "inputs": [{"url": u} for u in SAMPLE_IMAGE_URLS],
}

print("Sending 5 images to Content Understanding in one batch call, this takes about 15 to 30 seconds...")
# YOUR CODE: POST the batch payload via
# analyze_response = requests.post(
#     analyze_url,
#     headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
#     json=analyze_payload,
#     timeout=120,
# )

if analyze_response.status_code >= 400:
    print(f"Batch analyze failed: {analyze_response.status_code}")
    print(analyze_response.text)
    raise SystemExit(1)

batch_body = analyze_response.json()
records = batch_body.get("results") or batch_body.get("outputs") or []
if len(records) != len(SAMPLE_IMAGE_URLS):
    print(f"Expected {len(SAMPLE_IMAGE_URLS)} records, got {len(records)}.")

IMAGE_RECORDS = []
for url, rec in zip(SAMPLE_IMAGE_URLS, records):
    fields = rec.get("fields") or rec
    IMAGE_RECORDS.append({
        "image_url": url,
        "caption": fields.get("caption", ""),
        "alt_text": fields.get("alt_text", ""),
        "confidence": float(fields.get("confidence", 0.0)),
        "source": "content_understanding",
    })

print(f"Got {len(IMAGE_RECORDS)} image records.")
for r in IMAGE_RECORDS:
    print(f"  url={r['image_url'].split('/')[-1]:<14} "
          f"conf={r['confidence']:.2f} caption={r['caption']!r:.60}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Exactly 5 image records; every record has caption (>=10 chars),
# alt_text (>=5 chars), and a numeric confidence in [0, 1].

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        if not isinstance(IMAGE_RECORDS, list) or len(IMAGE_RECORDS) != 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"IMAGE_RECORDS has "
                    f"{len(IMAGE_RECORDS) if isinstance(IMAGE_RECORDS, list) else 'no'} "
                    f"entries, expected 5."
                ),
            )

        for i, r in enumerate(IMAGE_RECORDS, start=1):
            for field in ("image_url", "caption", "alt_text", "confidence", "source"):
                if field not in r:
                    return CheckpointResult(
                        status="fail",
                        error_reason=f"Record {i} is missing field {field!r}.",
                    )
            if not isinstance(r["caption"], str) or len(r["caption"]) < 10:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Record {i} caption is too short or wrong type: {r['caption']!r}. "
                        f"Expected a non-empty string of at least 10 characters."
                    ),
                )
            if not isinstance(r["alt_text"], str) or len(r["alt_text"]) < 5:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Record {i} alt_text is too short or wrong type: {r['alt_text']!r}."
                    ),
                )
            try:
                c = float(r["confidence"])
            except (TypeError, ValueError):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Record {i} confidence is not numeric: {r['confidence']!r}",
                )
            if c < 0.0 or c > 1.0:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Record {i} confidence={c} is outside [0, 1]. "
                        f"Content Understanding emits confidence on a 0 to 1 scale."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

If the response status is 401 or 403, the AAD token does not have data-plane access; confirm the AI Services account was created with `custom_sub_domain_name` (required for AAD auth on the data plane). If records are returned but fields are empty, the analyzer schema's `required` array did not propagate; PUT the analyzer again.

</details>

<details><summary>Hint 2 (direction)</summary>

One call: `analyze_response = requests.post(analyze_url, headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"}, json=analyze_payload, timeout=120)`. The `analyze_url`, `token`, and `analyze_payload` are all prepared above. The 120-second timeout accommodates the batch's 15-30 second analysis time.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
analyze_response = requests.post(
    analyze_url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json=analyze_payload,
    timeout=120,
)
```

</details>

**Wallet check.** Five images at about a tenth of a cent each is half a cent total. The HEAD checks in setup cost nothing. Coffee is still ahead by a factor of about 1000.

## Task 3: Dispatch low-confidence images to GPT-4o multimodal and replace the records

The fallback path is the production-grade pattern for caption pipelines. If Content Understanding emits a high-confidence record, you keep it. If confidence is below the threshold, you re-caption the image via GPT-4o's image-input path and replace the record. The lab artificially raises the threshold to 0.99 so every image triggers the fallback (production should use 0.7).

Build it in this order:

1. For every record where `confidence < CONFIDENCE_THRESHOLD`, call GPT-4o with the image URL.
2. The GPT-4o image-input shape: `messages=[{"role": "user", "content": [{"type": "text", "text": "..."}, {"type": "image_url", "image_url": {"url": url}}]}]`. Request structured JSON output with `response_format={"type": "json_object"}`.
3. Parse the JSON, replace the IMAGE_RECORDS entry with the GPT-4o output, mark `source="gpt4o_fallback"`, and set `confidence=1.0` (the fallback path is by definition the high-confidence path).

**Trap to avoid: raw bytes or base64 in the wrong place.** GPT-4o accepts image URLs OR base64-encoded data URLs. Passing raw bytes returns a 400. The lab uses URLs; the asset HEAD check in setup confirms they are reachable.

In [ ]:
# NBVAL_SKIP
# Task 3: GPT-4o multimodal fallback for low-confidence images.

token_provider = get_bearer_token_provider(
    AZURE_CREDENTIAL, "https://cognitiveservices.azure.com/.default",
)
aoai_client = AzureOpenAI(
    azure_endpoint=AOAI_ACCOUNT_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version="2024-08-01-preview",
)

fallback_prompt = (
    "Describe this product image for an accessibility-first ecommerce site. "
    'Return JSON exactly in this shape: {"caption": "<one-sentence>", '
    '"alt_text": "<concise alt text for screen readers>"}. '
    "Keep alt_text under 120 characters."
)

fallback_count = 0
for idx, rec in enumerate(IMAGE_RECORDS):
    if rec["confidence"] >= CONFIDENCE_THRESHOLD:
        continue

    print(f"Image {idx + 1} scored {rec['confidence']:.2f}, retrying via GPT-4o multimodal...")
    # YOUR CODE: Call GPT-4o with the image URL via
    # gpt4o_resp = aoai_client.chat.completions.create(
    #     model=CHAT_DEPLOYMENT_NAME,
    #     messages=[{
    #         "role": "user",
    #         "content": [
    #             {"type": "text", "text": fallback_prompt},
    #             {"type": "image_url", "image_url": {"url": rec["image_url"]}},
    #         ],
    #     }],
    #     response_format={"type": "json_object"},
    # )

    raw = gpt4o_resp.choices[0].message.content or "{}"
    parsed = json.loads(raw)
    IMAGE_RECORDS[idx] = {
        "image_url": rec["image_url"],
        "caption": parsed.get("caption", ""),
        "alt_text": parsed.get("alt_text", ""),
        "confidence": 1.0,
        "source": "gpt4o_fallback",
    }
    fallback_count += 1

print()
print(f"Fallback dispatched {fallback_count} time(s).")
for r in IMAGE_RECORDS:
    print(f"  source={r['source']:<22} conf={r['confidence']:.2f} caption={r['caption']!r:.60}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: At least one fallback dispatch happened; every record has a
# source field of either 'content_understanding' or 'gpt4o_fallback';
# fallback records have valid JSON-shaped caption and alt_text fields.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        if not IMAGE_RECORDS:
            return CheckpointResult(
                status="fail",
                error_reason="IMAGE_RECORDS is empty. Re-run Task 2 and Task 3.",
            )

        sources_seen = {r.get("source") for r in IMAGE_RECORDS}
        if "gpt4o_fallback" not in sources_seen:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No record has source='gpt4o_fallback'. The fallback branch did not fire. "
                    f"Check that CONFIDENCE_THRESHOLD ({CONFIDENCE_THRESHOLD}) is above the "
                    f"confidence scores in IMAGE_RECORDS; the lab sets 0.99 to guarantee at "
                    f"least one fallback dispatch."
                ),
            )

        for i, r in enumerate(IMAGE_RECORDS, start=1):
            src = r.get("source")
            if src not in ("content_understanding", "gpt4o_fallback"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Record {i} source is {src!r}, expected 'content_understanding' or 'gpt4o_fallback'."
                    ),
                )
            if src == "gpt4o_fallback":
                if not r.get("caption") or len(r["caption"]) < 5:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"gpt4o_fallback record {i} has a missing or too-short caption: "
                            f"{r.get('caption')!r}. Check that the JSON response was parsed "
                            f"and the field name matches."
                        ),
                    )
                if not r.get("alt_text") or len(r["alt_text"]) < 5:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"gpt4o_fallback record {i} has a missing or too-short alt_text: "
                            f"{r.get('alt_text')!r}."
                        ),
                    )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

If the validator says no record has `source='gpt4o_fallback'`, the fallback branch did not fire; `CONFIDENCE_THRESHOLD` is below the actual confidence scores. The lab pins it at 0.99 so at least one record falls under. If the call returns 400, the image-input shape is wrong (raw bytes instead of url).

</details>

<details><summary>Hint 2 (direction)</summary>

One call. The shape is `aoai_client.chat.completions.create(model=CHAT_DEPLOYMENT_NAME, messages=[{"role": "user", "content": [<text part>, <image_url part>]}], response_format={"type": "json_object"})`. The text part is `{"type": "text", "text": fallback_prompt}`. The image part is `{"type": "image_url", "image_url": {"url": rec["image_url"]}}`. JSON mode prevents Markdown fences around the JSON.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
gpt4o_resp = aoai_client.chat.completions.create(
    model=CHAT_DEPLOYMENT_NAME,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": fallback_prompt},
            {"type": "image_url", "image_url": {"url": rec["image_url"]}},
        ],
    }],
    response_format={"type": "json_object"},
)
```

</details>

**Wallet check.** Each GPT-4o image-input call is roughly 1,000 input + 150 output tokens at $2.50 + $10 per 1M = $0.0025 + $0.0015 = $0.004 per image. Five fallbacks lands around two cents. The bill still rounds to under a nickel for the whole image phase. Coffee is now about 100x ahead.

## Task 4: Analyze a 10-second product video and append per-shot records

Video analysis is async. You submit the video to the same analyzer's `:analyze` endpoint, get back an `operationId`, then poll a status endpoint until `Succeeded`. The result is a list of shots, each with `start_time`, `end_time`, `caption`, `alt_text`, and `confidence`.

Build it in this order:

1. POST the video URL to `{endpoint}/contentunderstanding/analyzers/{ANALYZER_ID}:analyze?api-version={version}` with `Operation-Location` header captured from the response.
2. Poll the `Operation-Location` URL with the same AAD token until the body's `status` is `"Succeeded"` (or `"Failed"`). Cap at 60 seconds.
3. Parse the result, build `VIDEO_RECORDS` with the same shape as `IMAGE_RECORDS` but with `media_type="video_shot"`, `start_time`, `end_time`.
4. Concatenate `IMAGE_RECORDS` and `VIDEO_RECORDS` into the final `COMBINED_ARTIFACT`. Add `media_type="image"` to image records.

In [ ]:
# NBVAL_SKIP
# Task 4: Async video analysis via the analyzer's :analyze endpoint.

token = AZURE_CREDENTIAL.get_token("https://cognitiveservices.azure.com/.default").token
video_analyze_url = (
    f"{AI_SERVICES_ENDPOINT}contentunderstanding/analyzers/{ANALYZER_ID}:analyze"
    f"?api-version={CONTENT_UNDERSTANDING_API_VERSION}"
)
video_payload = {"inputs": [{"url": SAMPLE_VIDEO_URL, "inputMode": "video"}]}

print("Submitting the 10-second video clip, this takes about 60 seconds...")
submit_resp = requests.post(
    video_analyze_url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json=video_payload,
    timeout=60,
)
if submit_resp.status_code >= 400:
    print(f"Video submit failed: {submit_resp.status_code} {submit_resp.text}")
    raise SystemExit(1)

# The operation_location header is the polling endpoint.
operation_location = submit_resp.headers.get("Operation-Location") or submit_resp.headers.get("operation-location")
if not operation_location:
    # Some preview API versions return the operation_location in the JSON body
    # instead of the header. Fall back to that path.
    body = submit_resp.json()
    operation_location = body.get("operationLocation") or body.get("operation_location")
if not operation_location:
    print("Could not find the Operation-Location header on the video submit response.")
    print(f"Headers: {dict(submit_resp.headers)}")
    raise SystemExit(1)

print(f"Polling {operation_location} for completion...")
video_result_body = None
for attempt in range(20):
    # YOUR CODE: Poll the operation_location URL via
    # poll_resp = requests.get(
    #     operation_location,
    #     headers={"Authorization": f"Bearer {token}"},
    #     timeout=30,
    # )
    if poll_resp.status_code >= 400:
        print(f"Poll failed: {poll_resp.status_code} {poll_resp.text}")
        raise SystemExit(1)
    body = poll_resp.json()
    status_field = body.get("status") or body.get("Status") or ""
    if status_field.lower() == "succeeded":
        video_result_body = body
        break
    if status_field.lower() == "failed":
        print(f"Video analysis failed: {body}")
        raise SystemExit(1)
    print(f"  Attempt {attempt + 1}: status={status_field}, waiting 5 seconds...")
    time.sleep(5)

if video_result_body is None:
    print("Video analysis exceeded the 100-second polling ceiling.")
    raise SystemExit(1)

# Parse shot segments. The result schema nests segments under
# result.contents[].fields per the Content Understanding 2024-12-01-preview
# contract.
VIDEO_RECORDS = []
contents = (
    video_result_body.get("result", {}).get("contents")
    or video_result_body.get("contents")
    or []
)
for seg in contents:
    fields = seg.get("fields") or seg
    VIDEO_RECORDS.append({
        "video_url": SAMPLE_VIDEO_URL,
        "start_time": float(seg.get("startTime", seg.get("start_time", 0)) or 0),
        "end_time": float(seg.get("endTime", seg.get("end_time", 0)) or 0),
        "caption": fields.get("caption", ""),
        "alt_text": fields.get("alt_text", ""),
        "confidence": float(fields.get("confidence", 0.0)),
        "source": "content_understanding",
        "media_type": "video_shot",
    })

print(f"Got {len(VIDEO_RECORDS)} video shot record(s).")
for r in VIDEO_RECORDS:
    print(f"  start={r['start_time']:.1f}s end={r['end_time']:.1f}s "
          f"conf={r['confidence']:.2f} caption={r['caption']!r:.60}")

# Build the combined artifact. Image records get media_type='image'.
COMBINED_ARTIFACT = []
for r in IMAGE_RECORDS:
    COMBINED_ARTIFACT.append({**r, "media_type": "image"})
COMBINED_ARTIFACT.extend(VIDEO_RECORDS)
print()
print(f"Combined artifact: {len(COMBINED_ARTIFACT)} total record(s) "
      f"({sum(1 for r in COMBINED_ARTIFACT if r['media_type'] == 'image')} image, "
      f"{sum(1 for r in COMBINED_ARTIFACT if r['media_type'] == 'video_shot')} video_shot)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: At least 2 video shot records; each has start_time, end_time,
# caption, alt_text, confidence, and media_type='video_shot'. The combined
# artifact contains both image and video_shot records.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        if not isinstance(VIDEO_RECORDS, list) or len(VIDEO_RECORDS) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"VIDEO_RECORDS has {len(VIDEO_RECORDS) if isinstance(VIDEO_RECORDS, list) else 'no'} "
                    f"entries, expected at least 2. The analyzer chunks 10 seconds into 2-4 shots "
                    f"in default mode; if you got <2 the polling may have terminated early."
                ),
            )

        for i, r in enumerate(VIDEO_RECORDS, start=1):
            for field in ("start_time", "end_time", "caption", "alt_text", "confidence", "media_type"):
                if field not in r:
                    return CheckpointResult(
                        status="fail",
                        error_reason=f"Video record {i} missing {field!r}.",
                    )
            if r["media_type"] != "video_shot":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Video record {i} media_type is {r['media_type']!r}, "
                        f"expected 'video_shot'."
                    ),
                )
            if r["end_time"] <= r["start_time"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Video record {i} has end_time={r['end_time']} <= "
                        f"start_time={r['start_time']}."
                    ),
                )

        if not isinstance(COMBINED_ARTIFACT, list):
            return CheckpointResult(
                status="fail",
                error_reason="COMBINED_ARTIFACT is not a list.",
            )
        media_types = {r.get("media_type") for r in COMBINED_ARTIFACT}
        if "image" not in media_types:
            return CheckpointResult(
                status="fail",
                error_reason="COMBINED_ARTIFACT has no records with media_type='image'.",
            )
        if "video_shot" not in media_types:
            return CheckpointResult(
                status="fail",
                error_reason="COMBINED_ARTIFACT has no records with media_type='video_shot'.",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

If the polling loop runs the full 20 attempts and the operation still says `Running`, the video is taking longer than usual; bump the ceiling to 120 attempts. If it says `Failed`, check the body's `error` field for a code; common causes are unreadable URLs and unsupported codecs.

</details>

<details><summary>Hint 2 (direction)</summary>

One call inside the loop: `poll_resp = requests.get(operation_location, headers={"Authorization": f"Bearer {token}"}, timeout=30)`. The polling URL and token are already prepared; the loop handles the status check around your call.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
poll_resp = requests.get(
    operation_location,
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)
```

</details>

**Wallet check.** Ten seconds of video at about a tenth of a cent per second is one cent. Five images plus 10 seconds of video plus five GPT-4o fallback calls totals about $0.025. The Foundry stack and the AI Services account cost $0 at rest. Coffee remains comfortably ahead.

## Cleanup

Time to tear it all down. The cell below runs the manifest in reverse-creation order: GPT-4o deployment, AI Services account (which removes the analyzer schema), Foundry project, hub, resource group. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST.

import sys

for r in CLEANUP_MANIFEST:
    if AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

# Patch resource.extra with constants resolved during the lab so the
# Azure adapter sees account_name, service_name, and project_endpoint
# at cleanup time (manifest is built before those constants are set).
for r in CLEANUP_MANIFEST:
    if r.type in ("aoai_deployment", "aoai_rai_policy") and AOAI_ACCOUNT_NAME:
        r.extra = {"account_name": AOAI_ACCOUNT_NAME}
    elif r.type == "search_index" and "SEARCH_SERVICE_NAME" in globals() and SEARCH_SERVICE_NAME:
        r.extra = {"service_name": SEARCH_SERVICE_NAME}
    elif r.type == "foundry_agent" and "PROJECT_ENDPOINT" in globals() and PROJECT_ENDPOINT:
        r.extra = {"project_endpoint": PROJECT_ENDPOINT}

result = run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about five cents in a clean run, up to thirty cents with heavy debugging.** Content Understanding is cheap; the GPT-4o image-input fallback is the line item that adds up. The Foundry hub, project, AI Services account, and Standard deployment carry no hourly fee at zero traffic; the resource group delete is the safety net. Check Azure Cost Management in 24 hours to confirm.

## Reflection

These are not graded. They are for you.

1. Content Understanding's structured-output analyzer surface is convenient when you know the fields up front. GPT-4o multimodal is more flexible but more expensive and less consistent. Walk through what signal would push you to use Content Understanding for production accessibility captioning versus GPT-4o, and how you would measure the trade-off after a 30-day pilot.

2. The video output produced per-shot segments with start and end times. For an accessibility caption track (SRT or WebVTT), what additional metadata would you need beyond what the analyzer produced, and where in Azure would you store the captions to surface them to the video player?

## Exam-Style Practice

**Question 1 (Domain 3, Content Understanding versus Video Indexer):**

A team is choosing between Azure AI Content Understanding and Azure AI Video Indexer for analyzing a 30-minute video. The team needs structured per-shot caption JSON and shot-level alt-text. Both services can do this. Which signal would push the team toward Content Understanding?

A. Video Indexer has higher per-minute throughput.

B. Content Understanding integrates with Foundry Tools as a first-class analyzer and produces structured JSON output via a declarative analyzer schema; Video Indexer's API surface is richer but the output is less directly consumable by an LLM-grounded pipeline.

C. Video Indexer requires a Microsoft 365 license; Content Understanding does not.

D. Content Understanding is free for the first 100 hours of video per month; Video Indexer is pay-as-you-go from the first minute.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: both services handle large video workloads at production scale.
- B is correct: Content Understanding is purpose-built for LLM-pipeline integration with declarative analyzer schemas and structured outputs. Video Indexer is a richer media-analysis surface with insights like topic detection and OCR. For an LLM-grounded structured-output pipeline, Content Understanding is the Microsoft-recommended choice as of March 2026.
- C is wrong: neither service requires a Microsoft 365 license.
- D is wrong: there is no 100-hour-per-month free tier on Content Understanding. Both bill per unit of media analyzed.

</details>

**Question 2 (Domain 3, fallback dispatch architecture):**

A captioning pipeline uses Content Understanding for batch image analysis with a confidence threshold of 0.7. Below that threshold, the pipeline falls back to GPT-4o multimodal. The team wants to track the fraction of images that hit the fallback path over time and alert if it exceeds 15% of weekly volume. Which observability path fits?

A. Emit a custom OpenTelemetry counter `fallback_dispatches_total` with `source` dimension and an alert rule on `fallback_dispatches_total / total_dispatches_total > 0.15` over a 7-day window.

B. Send the per-image JSON records to Cosmos DB and run a daily aggregation query.

C. Read the Content Understanding billing line items monthly and reason about the ratio.

D. Run the `RelevanceEvaluator` from `azure-ai-evaluation` over the captions.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: this is the standard observability pattern for branch-dispatch metrics. OpenTelemetry counters with dimensions land in App Insights / Log Analytics and support alert rules via Azure Monitor. This is the AI-103 documented observability path.
- B is wrong: Cosmos DB works but is overkill for a counter metric and does not provide native alerting.
- C is wrong: billing is delayed by 24+ hours and is a coarse signal. Not actionable for a 15% threshold.
- D is wrong: `RelevanceEvaluator` measures caption-to-image relevance, not fallback dispatch rate.

</details>